In [20]:
# # Weather Classification with GNN (Graph-Level Task)

# This notebook demonstrates how to build a Graph Neural Network (GNN) to classify global weather states into three categories: **Sunny**, **Cloudy**, and **Rainy**.

# ## Task Type: Graph-Level Classification
# - Each graph represents a global weather snapshot at one timestep
# - 2,048 nodes (grid cells covering the globe)
# - Goal: Classify the entire graph into one weather category

# ## Architecture:
# 1. **Embedding Layer**: Projects 4 raw features → 64-dim embeddings
# 2. **5 GNN Layers**: Message passing across spatial neighbors (5 hops)
# 3. **Global Pooling**: Aggregates all nodes → single graph representation
# 4. **MLP Classifier**: Graph embedding → logits → softmax (3 classes)


In [21]:
import os
import xarray as xr

local_path = "wb2_64x32_weather_2020-2023.zarr"

if os.path.exists(local_path):
    print(f"✓ Local dataset already exists at '{local_path}'. Skipping download.")
else:
    store = "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-64x32_equiangular_conservative.zarr"
    ds = xr.open_zarr(store, chunks={})

    # Subset to just 2020-2023 to keep size manageable (~0.14 GB in memory)
    ds_subset = ds.sel(time=slice("2020-01-01", "2023-12-31"))

    print(f"Original dataset: {len(ds.time)} timesteps")
    print(f"Subset dataset: {len(ds_subset.time)} timesteps")

    # Select variables for weather prediction
    sub = xr.Dataset({
        "t2m": ds_subset["2m_temperature"],                    # 2m temperature
        "tp6h": ds_subset["total_precipitation_6hr"],          # 6-hour precipitation
        "q850": ds_subset["specific_humidity"].sel(level=850), # specific humidity at 850 hPa
        "rh": ds_subset["relative_humidity"].sel(level=850),   # relative humidity at 850 hPa
        "z_sfc": ds_subset["geopotential_at_surface"],         # surface geopotential (elevation)
    })

    # Rechunk to avoid chunk overlap issues when saving
    print("\nRechunking data for proper zarr storage...")
    sub = sub.chunk({"time": 100, "longitude": 64, "latitude": 32})

    print("Saving weather data subset (this will take a few minutes)...")
    sub.to_zarr(local_path, mode="w")
    print(f"Done! Data saved to: {local_path}")


✓ Local dataset already exists at 'wb2_64x32_weather_2020-2023.zarr'. Skipping download.


In [22]:
import numpy as np

# Load the downloaded dataset
ds = xr.open_zarr("wb2_64x32_weather_2020-2023.zarr")

print("=" * 70)
print("WEATHER DATA SUMMARY")
print("=" * 70)

print("\n📊 DIMENSIONS:")
print(f"  • Time steps: {len(ds.time)} (6-hourly data)")
print(f"  • Latitude points: {len(ds.latitude)}")
print(f"  • Longitude points: {len(ds.longitude)}")
print(f"  • Total grid cells (nodes for GNN): {len(ds.latitude) * len(ds.longitude)}")

print("\n📅 TIME RANGE:")
print(f"  • Start: {ds.time.values[0]}")
print(f"  • End: {ds.time.values[-1]}")
print(f"  • Duration: ~{len(ds.time) / (4 * 365):.1f} years")

print("\n🗺️  SPATIAL COVERAGE:")
print(f"  • Latitude range: {ds.latitude.values.min():.2f}° to {ds.latitude.values.max():.2f}°")
print(f"  • Longitude range: {ds.longitude.values.min():.2f}° to {ds.longitude.values.max():.2f}°")

print("\n🌡️  VARIABLES (Features):")
for var in ds.data_vars:
    shape = ds[var].shape
    size_mb = ds[var].nbytes / (1024**2)
    print(f"  • {var:10s}: shape {shape}, size ~{size_mb:.1f} MB")

print("\n💾 TOTAL DATASET SIZE (uncompressed):")
print(f"  • Estimated: ~{sum(ds[var].nbytes for var in ds.data_vars) / (1024**3):.2f} GB")

print("\n📈 SAMPLE DATA (first timestep, first location):")
for var in ds.data_vars:
    if 'time' in ds[var].dims:
        val = float(ds[var].isel(time=0, latitude=0, longitude=0).values)
    else:
        val = float(ds[var].isel(latitude=0, longitude=0).values)
    print(f"  • {var:10s}: {val:.4f}")

print("\n🎯 TASK CONTEXT:")
print(f"  • Node features: 4 (t2m, q850, rh, z_sfc)")
print(f"  • Target to predict: 1 (tp6h - precipitation)")
print(f"  • Graph size: 2,048 nodes, 64×32 grid")

print("\n" + "=" * 70)


WEATHER DATA SUMMARY

📊 DIMENSIONS:
  • Time steps: 4424 (6-hourly data)
  • Latitude points: 32
  • Longitude points: 64
  • Total grid cells (nodes for GNN): 2048

📅 TIME RANGE:
  • Start: 2020-01-01T00:00:00.000000000
  • End: 2023-01-10T18:00:00.000000000
  • Duration: ~3.0 years

🗺️  SPATIAL COVERAGE:
  • Latitude range: -87.19° to 87.19°
  • Longitude range: 0.00° to 354.38°

🌡️  VARIABLES (Features):
  • q850      : shape (4424, 64, 32), size ~34.6 MB
  • rh        : shape (4424, 64, 32), size ~34.6 MB
  • t2m       : shape (4424, 64, 32), size ~34.6 MB
  • tp6h      : shape (4424, 64, 32), size ~34.6 MB
  • z_sfc     : shape (64, 32), size ~0.0 MB

💾 TOTAL DATASET SIZE (uncompressed):
  • Estimated: ~0.14 GB

📈 SAMPLE DATA (first timestep, first location):
  • q850      : 0.0006
  • rh        : 0.3678
  • t2m       : 248.4972
  • tp6h      : 0.0000
  • z_sfc     : 25481.7461

🎯 TASK CONTEXT:
  • Node features: 4 (t2m, q850, rh, z_sfc)
  • Target to predict: 1 (tp6h - precipitat

In [23]:
import subprocess

# Compressed size on disk
result = subprocess.run(["du", "-sh", "wb2_64x32_weather_2020-2023.zarr"], capture_output=True, text=True)
disk_size_human = result.stdout.split()[0]
disk_size_mb = float(disk_size_human.replace('M', '')) if disk_size_human.endswith('M') else None

# Uncompressed size in memory
ds = xr.open_zarr("wb2_64x32_weather_2020-2023.zarr")
total_bytes = sum(ds[var].nbytes for var in ds.data_vars)
uncompressed_mb = total_bytes / (1024**2)

print("=" * 70)
print("STORAGE COMPARISON")
print("=" * 70)

print(f"\n💾 COMPRESSED (on disk with zarr):")
print(f"  • Size: {disk_size_human}")
if disk_size_mb:
    print(f"  • ~{disk_size_mb:.1f} MB")

print(f"\n📦 UNCOMPRESSED (loaded in memory):")
print(f"  • {uncompressed_mb:.1f} MB ({uncompressed_mb/1024:.3f} GB)")

if disk_size_mb:
    compression_ratio = uncompressed_mb / disk_size_mb
    print(f"\n📊 COMPRESSION RATIO:")
    print(f"  • {compression_ratio:.2f}x smaller on disk")
    print(f"  • Saved: {uncompressed_mb - disk_size_mb:.1f} MB")

print("\n📝 BREAKDOWN BY VARIABLE (uncompressed):")
for var in ds.data_vars:
    size_mb = ds[var].nbytes / (1024**2)
    pct = (ds[var].nbytes / total_bytes) * 100
    print(f"  • {var:10s}: {size_mb:6.1f} MB ({pct:5.1f}%)")

print("\n" + "=" * 70)


STORAGE COMPARISON

💾 COMPRESSED (on disk with zarr):
  • Size: 116M
  • ~116.0 MB

📦 UNCOMPRESSED (loaded in memory):
  • 138.3 MB (0.135 GB)

📊 COMPRESSION RATIO:
  • 1.19x smaller on disk
  • Saved: 22.3 MB

📝 BREAKDOWN BY VARIABLE (uncompressed):
  • q850      :   34.6 MB ( 25.0%)
  • rh        :   34.6 MB ( 25.0%)
  • t2m       :   34.6 MB ( 25.0%)
  • tp6h      :   34.6 MB ( 25.0%)
  • z_sfc     :    0.0 MB (  0.0%)



In [24]:
# Ensure required packages are installed
import sys
import subprocess

packages_to_install = []

try:
    import torch
    print(f"✓ PyTorch {torch.__version__} installed")
except ImportError:
    packages_to_install.append("torch")

try:
    import torch_geometric
    print(f"✓ PyTorch Geometric {torch_geometric.__version__} installed")
except ImportError:
    packages_to_install.extend(["torch-geometric", "torch-scatter", "torch-sparse"])

try:
    import sklearn
    print(f"✓ scikit-learn {sklearn.__version__} installed")
except ImportError:
    packages_to_install.append("scikit-learn")

try:
    import seaborn
    print(f"✓ seaborn {seaborn.__version__} installed")
except ImportError:
    packages_to_install.append("seaborn")

try:
    from tqdm.auto import tqdm
    print("✓ tqdm installed")
except ImportError:
    packages_to_install.append("tqdm")

if packages_to_install:
    print(f"\nInstalling: {', '.join(packages_to_install)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages_to_install + ["-q"])
    print("✓ Installation complete!")
else:
    print("\n✓ All packages already installed!")


✓ PyTorch 2.5.1 installed
✓ PyTorch Geometric 2.6.1 installed
✓ scikit-learn 1.3.1 installed
✓ seaborn 0.13.2 installed
✓ tqdm installed

✓ All packages already installed!


# Weather Prediction GNN - Architecture Plan

## 🎯 Task: Node-Level Temporal Prediction
**Predict precipitation at each grid cell at time t+1, given weather features at time t**

## 📊 Architecture Pipeline

```
Time t                    Time t+1
  ↓                          ↓
[Features]            [Precipitation]
  t2m, q850,              tp6h
  rh, z_sfc              (target)
    ↓
[Normalize]
    ↓
[Feature Encoder]
  MLP: 4→32→64
    ↓
[GNN Layers] ←→ [Graph Structure]
  5× GCN           2048 nodes
  + Residual       8-way edges
  + LayerNorm
  + Dropout
    ↓
[Decoder]
  MLP: 64→32→1
    ↓
[Predictions]
  ŷ_i for each node i
    ↓
[Loss: MSE]
  Compare with tp6h(t+1)
```

## 🏗️ Key Components

1. **Graph Structure**
   - Nodes: 2048 grid cells (64×32)
   - Edges: 8-way spatial connectivity
   - Node features: 4D (temperature, humidity×2, elevation)

2. **GNN Architecture**
   - Encoder: MLP (4 → 64 dims)
   - 5 GCN layers with residuals
   - Decoder: MLP (64 → 1 prediction)

3. **Training**
   - Input: Features at time t
   - Output: Precipitation at time t+1
   - Loss: MSE (regression)
   - Metrics: MAE, RMSE, R²

See `graph.py` for detailed architectural plan!


In [25]:
import torch
import numpy as np
import xarray as xr

print("=" * 70)
print("STEP 1: DATA PREPARATION")
print("=" * 70)

# Load dataset (ensure fully in-memory and NaN-free)
ds = xr.open_zarr("wb2_64x32_weather_2020-2023.zarr").load()
ds = ds.fillna(0.0)

# Grid info
lats = ds.latitude.values
lons = ds.longitude.values
n_lat, n_lon = len(lats), len(lons)
n_nodes = n_lat * n_lon

print(f"\n✓ Loaded data: {len(ds.time)} timesteps, {n_nodes} nodes")

# Build grid graph (4-neighborhood with periodic longitude)
edge_list = []
for i in range(n_lon):
    for j in range(n_lat):
        node_idx = i * n_lat + j
        # Right neighbor (wrap around)
        right_i = (i + 1) % n_lon
        right_idx = right_i * n_lat + j
        edge_list.append([node_idx, right_idx])
        # Down neighbor (no wrap)
        if j + 1 < n_lat:
            down_idx = i * n_lat + (j + 1)
            edge_list.append([node_idx, down_idx])

edge_index = torch.tensor(edge_list, dtype=torch.long).t()
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)  # make bidirectional

print(f"✓ Built graph: {edge_index.shape[1]} edges (bidirectional)")

# Create graph-level labels from precipitation percentiles
tp_data = ds['tp6h'].values  # (time, lon, lat)
avg_precips = np.nanmean(tp_data, axis=(1, 2))
avg_precips = np.nan_to_num(avg_precips, nan=0.0)

print(f"\n📊 Precipitation Distribution:")
print(f"  • Min: {np.nanmin(avg_precips):.6f}")
print(f"  • 25th percentile: {np.nanpercentile(avg_precips, 25):.6f}")
print(f"  • Median: {np.nanmedian(avg_precips):.6f}")
print(f"  • 75th percentile: {np.nanpercentile(avg_precips, 75):.6f}")
print(f"  • Max: {np.nanmax(avg_precips):.6f}")

threshold_low = np.nanpercentile(avg_precips, 33.3)
threshold_high = np.nanpercentile(avg_precips, 66.6)

labels = np.zeros(len(avg_precips), dtype=np.int64)
labels[avg_precips >= threshold_high] = 2  # Rainy
mid_mask = (avg_precips >= threshold_low) & (avg_precips < threshold_high)
labels[mid_mask] = 1  # Cloudy (medium precipitation)
# Remaining default 0 (Sunny)

print(f"\n✓ Created balanced labels using percentiles:")
print(f"  • Sunny (0): {(labels == 0).sum()} samples ({(labels == 0).mean()*100:.1f}%) | precip < {threshold_low:.6f}")
print(f"  • Cloudy (1): {(labels == 1).sum()} samples ({(labels == 1).mean()*100:.1f}%) | {threshold_low:.6f} ≤ precip < {threshold_high:.6f}")
print(f"  • Rainy (2): {(labels == 2).sum()} samples ({(labels == 2).mean()*100:.1f}%) | precip ≥ {threshold_high:.6f}")

print("\n" + "=" * 70)


STEP 1: DATA PREPARATION

✓ Loaded data: 4424 timesteps, 2048 nodes
✓ Built graph: 8064 edges (bidirectional)

📊 Precipitation Distribution:
  • Min: 0.000511
  • 25th percentile: 0.000584
  • Median: 0.000608
  • 75th percentile: 0.000631
  • Max: 0.000728

✓ Created balanced labels using percentiles:
  • Sunny (0): 1473 samples (33.3%) | precip < 0.000592
  • Cloudy (1): 1473 samples (33.3%) | 0.000592 ≤ precip < 0.000623
  • Rainy (2): 1478 samples (33.4%) | precip ≥ 0.000623



In [ ]:
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

print("=" * 70)
print("STEP 2: CREATE GRAPH DATASET")
print("=" * 70)

# Extract variables
t2m = ds['t2m'].values
q850 = ds['q850'].values
rh = ds['rh'].values
z_sfc = ds['z_sfc'].values

# Normalization helper
def normalize(arr, axis=None):
    mean = arr.mean(axis=axis, keepdims=True)
    std = arr.std(axis=axis, keepdims=True) + 1e-8
    return (arr - mean) / std

t2m_norm = normalize(t2m)
q850_norm = normalize(q850)
rh_norm = normalize(rh)
z_sfc_norm = normalize(z_sfc)

# Build torch geometric Data objects
graph_list = []
for t in range(len(ds.time)):
    feature_grid = np.stack([
        t2m_norm[t],
        q850_norm[t],
        rh_norm[t],
        z_sfc_norm
    ], axis=-1)  # (lat, lon, features)
    feature_grid = np.transpose(feature_grid, (1, 0, 2))  # (lon, lat, features)
    features = feature_grid.reshape(-1, 4)
    
    data = Data(
        x=torch.tensor(features, dtype=torch.float32),
        edge_index=edge_index,
        y=torch.tensor([labels[t]], dtype=torch.long)
    )
    graph_list.append(data)

print(f"\n✓ Created {len(graph_list)} graph snapshots")
print(f"  • Each graph: {graph_list[0].num_nodes} nodes, {graph_list[0].num_edges} edges")
print(f"  • Node features shape: {graph_list[0].x.shape}")

# Train / Val / Test split
indices = np.arange(len(graph_list))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=labels)
train_idx, val_idx = train_test_split(train_idx, test_size=0.15, random_state=42, stratify=labels[train_idx])

def gather_graphs(idxs):
    return [graph_list[i] for i in idxs]

train_graphs = gather_graphs(train_idx)
val_graphs = gather_graphs(val_idx)
test_graphs = gather_graphs(test_idx)

print(f"\n✓ Data split:")
print(f"  • Train: {len(train_graphs)} graphs")
print(f"  • Val: {len(val_graphs)} graphs")
print(f"  • Test: {len(test_graphs)} graphs")

# DataLoaders (batch multiple graphs)
batch_size = 32
train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=batch_size, shuffle=False)

print("\n" + "=" * 70)


STEP 2: CREATE GRAPH DATASET

✓ Created 4424 graph snapshots
  • Each graph: 2048 nodes, 8064 edges
  • Node features shape: torch.Size([2048, 4])

✓ Data split:
  • Train: 3008 graphs
  • Val: 531 graphs
  • Test: 885 graphs



/Users/gleb/miniconda3/envs/gnn_env/lib/python3.9/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [27]:
print("=" * 70)
print("DATA SPLIT CONFIRMATION")
print("=" * 70)

print(f"\n📊 Dataset Split:")
print(f"  • Total graphs: {len(graph_list)}")
print(f"  • Train: {len(train_graphs)} graphs ({len(train_graphs)/len(graph_list)*100:.1f}%)")
print(f"  • Val: {len(val_graphs)} graphs ({len(val_graphs)/len(graph_list)*100:.1f}%)")
print(f"  • Test: {len(test_graphs)} graphs ({len(test_graphs)/len(graph_list)*100:.1f}%)")

print(f"\n📦 Batch Information:")
print(f"  • Batch size: {batch_size}")
print(f"  • Train batches: {len(train_loader)}")
print(f"  • Val batches: {len(val_loader)}")
print(f"  • Test batches: {len(test_loader)}")

print(f"\n🎯 Class Distribution per split:")
for split_name, split_graphs in [("Train", train_graphs), ("Val", val_graphs), ("Test", test_graphs)]:
    labels_split = [g.y.item() for g in split_graphs]
    print(f"  {split_name:5s}: Sunny={sum(l==0 for l in labels_split):4d} | "
          f"Cloudy={sum(l==1 for l in labels_split):4d} | "
          f"Rainy={sum(l==2 for l in labels_split):4d}")

print("\n" + "=" * 70)


DATA SPLIT CONFIRMATION

📊 Dataset Split:
  • Total graphs: 4424
  • Train: 3008 graphs (68.0%)
  • Val: 531 graphs (12.0%)
  • Test: 885 graphs (20.0%)

📦 Batch Information:
  • Batch size: 32
  • Train batches: 94
  • Val batches: 17
  • Test batches: 28

🎯 Class Distribution per split:
  Train: Sunny=1001 | Cloudy=1002 | Rainy=1005
  Val  : Sunny= 177 | Cloudy= 177 | Rainy= 177
  Test : Sunny= 295 | Cloudy= 294 | Rainy= 296



In [28]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool

print("=" * 70)
print("STEP 3: BUILD GNN MODEL")
print("=" * 70)

class WeatherGNN(nn.Module):
    """Graph-level weather classifier."""
    def __init__(self, num_features, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Linear(num_features, hidden_dim)
        self.conv_layers = nn.ModuleList([
            GCNConv(hidden_dim, hidden_dim) for _ in range(5)
        ])
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.embedding(x))
        for conv in self.conv_layers:
            x = F.relu(conv(x, edge_index))
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        graph_repr = torch.cat([x_mean, x_max], dim=1)
        logits = self.mlp(graph_repr)
        return logits

num_features = 4
hidden_dim = 64
num_classes = 3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = WeatherGNN(num_features, hidden_dim, num_classes).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n✓ Model architecture ready:")
print(f"  • Input features per node: {num_features}")
print(f"  • Hidden dimension: {hidden_dim}")
print(f"  • GCN layers: {len(model.conv_layers)} (5 hops)")
print(f"  • Parameters: {num_params:,}")
print(f"  • Device: {device}")

print("\n" + "=" * 70)


STEP 3: BUILD GNN MODEL

✓ Model architecture ready:
  • Input features per node: 4
  • Hidden dimension: 64
  • GCN layers: 5 (5 hops)
  • Parameters: 29,571
  • Device: cpu



In [29]:
from tqdm.auto import tqdm

print("=" * 70)
print("STEP 4: TRAINING")
print("=" * 70)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()


def train_epoch(model, loader, optimizer, criterion, device, epoch_num=None, num_epochs=None):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    desc = f"Epoch {epoch_num}/{num_epochs}" if epoch_num else "Training"
    pbar = tqdm(loader, desc=desc, leave=False, ncols=100)
    for batch in pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch)
        loss = criterion(logits, batch.y)
        loss.backward()
        optimizer.step()
        
        batch_loss = loss.item()
        total_loss += batch_loss * batch.num_graphs
        preds = logits.argmax(dim=1)
        correct += (preds == batch.y).sum().item()
        total += batch.num_graphs
        
        pbar.set_postfix({
            "batch_loss": f"{batch_loss:.4f}",
            "avg_loss": f"{total_loss / total:.4f}",
            "acc": f"{correct / total:.4f}"
        })
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch)
            loss = criterion(logits, batch.y)
            total_loss += loss.item() * batch.num_graphs
            preds = logits.argmax(dim=1)
            correct += (preds == batch.y).sum().item()
            total += batch.num_graphs
    return total_loss / total, correct / total

num_epochs = 50
best_val_acc = 0.0
best_epoch = 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print("\n🚀 Training started...\n")

epoch_bar = tqdm(range(1, num_epochs + 1), desc="Epochs", unit="epoch")
for epoch in epoch_bar:
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device, epoch, num_epochs)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        torch.save(model.state_dict(), "best_weather_gnn.pt")
        best_flag = " ⭐ BEST"
    else:
        best_flag = ""
    
    epoch_bar.set_postfix({
        "train_loss": f"{train_loss:.4f}",
        "val_acc": f"{val_acc:.4f}",
        "best": f"{best_val_acc:.4f}"
    })
    
    print(f"Epoch {epoch:3d}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}{best_flag}")

print(f"\n✓ Training complete! Best validation accuracy: {best_val_acc:.4f} (Epoch {best_epoch})")

model.load_state_dict(torch.load("best_weather_gnn.pt"))
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"  • Test accuracy: {test_acc:.4f}")

print("\n" + "=" * 70)


STEP 4: TRAINING

🚀 Training started...



Epochs:   0%|          | 0/50 [00:00<?, ?epoch/s]

Epoch 1/50:   0%|                                                            | 0/94 [00:00<?, ?it/s]

Epoch   1/50 | Train Loss: 1.0733 | Train Acc: 0.3890 | Val Loss: 1.0197 | Val Acc: 0.4840 ⭐ BEST


Epoch 2/50:   0%|                                                            | 0/94 [00:00<?, ?it/s]

...

Epoch  36/50 | Train Loss: 0.9197 | Train Acc: 0.5356 | Val Loss: 0.9501 | Val Acc: 0.5235


Epoch 37/50:   0%|                                                           | 0/94 [00:00<?, ?it/s]

Epoch  37/50 | Train Loss: 0.9210 | Train Acc: 0.5273 | Val Loss: 0.9699 | Val Acc: 0.5292


Epoch 38/50:   0%|                                                           | 0/94 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train Loss', linewidth=2)
ax1.plot(history['val_loss'], label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history['train_acc'], label='Train Acc', linewidth=2)
ax2.plot(history['val_acc'], label='Val Acc', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training & Validation Accuracy')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Training curves saved to 'training_history.png'")
